In [18]:
import pandas as pd

# Cleaning the data
### 1. Characters df
##### Removing columns that are not needed

In [19]:
dfCharacters = pd.read_csv("../data/raw/characters.csv")

dfCharacters = dfCharacters.drop(columns=["url",
                                          "name_kanji",
                                          "image",
                                          "about"],
                                 errors="ignore")

### Normalizing
##### Favorites cast to int and set NaN to 0

In [20]:
dfCharacters.drop_duplicates()
dfCharacters["favorites"] = (
    pd.to_numeric(dfCharacters["favorites"], errors="coerce")
      .fillna(0)
      .astype(int)
)

##### ID set to int and removed Nan values

In [21]:
dfCharacters = dfCharacters[dfCharacters["character_mal_id"] != 0]
dfCharacters["character_mal_id"] = (
    pd.to_numeric(dfCharacters["character_mal_id"], errors="coerce")
        .fillna(-1)
        .astype(int)
)
dfCharacters = dfCharacters[dfCharacters["character_mal_id"] != -1]
dfCharacters

,character_mal_id,name,favorites
0,280386,Envi Mel Champagne,0
1,280354,Eleven,0
2,280353,Stud,0
3,280352,Judge,0
4,280339,Eiji Kurokawa,0
...,...,...,...
209958,282276,Farrah Van Dorothy,0
209959,282277,Harris Mead,0
209960,282278,Rob,0
209961,282281,Grimm,0


### 2. Nicknames df
##### Removed rows where nickname was empty or Nan, useless
##### Some Characters have multiple nicknames, so it's better to put them in the same row as a list

In [22]:
dfNicknames = pd.read_csv("../data/raw/character_nicknames.csv")

dfNicknames = dfNicknames[dfNicknames["nickname"].notna()]

dfNicknames = (
    dfNicknames
    .groupby("character_mal_id")["nickname"]
    .apply(list)
    .reset_index()
)

In [23]:
dfNicknames

,character_mal_id,nickname
0,3,"[Running Rock, Black Dog]"
1,5,"[Ichi-nii, Shinigami Daiko (Substitute Soul Re..."
2,7,[Hime]
3,11,"[Ed, Fullmetal Alchemist, Hagane no shounen, C..."
4,12,"[Al, Armored Alchemist]"
...,...,...
30262,282048,[Great Zuma]
30263,282159,"[Ling Long, Silvermoon]"
30264,282227,[Mei's Mother]
30265,282254,[Cyrano]


# Merging

In [24]:
dfCharacters = dfCharacters.merge(dfNicknames, on="character_mal_id", how = "left")
dfNicknames = None

In [25]:
dfCharacters['nickname'] = dfCharacters['nickname'].apply(lambda x: x if isinstance(x, list) else [])

### 3. AnimeWorks df

In [26]:
dfAnimeWorks = pd.read_csv("../data/raw/character_anime_works.csv")
dfCharacters = dfCharacters.merge(dfAnimeWorks, on="character_mal_id")
dfAnimeWorks = None
dfCharacters = dfCharacters.drop(columns=["name"]) # Redundant column, keeping the one with the comma
dfCharacters

,character_mal_id,favorites,nickname,anime_mal_id,character_name,role
0,280386,0,[],59846,"Champagne, Envi Mel",Supporting
1,280354,0,[],60071,Eleven,Supporting
2,280353,0,[],60071,Stud,Supporting
3,280352,0,[],60071,Judge,Supporting
4,280339,0,[],60531,"Kurokawa, Eiji",Supporting
...,...,...,...,...,...,...
238936,282276,0,[],3258,"Van Dorothy, Farrah",Supporting
238937,282277,0,[],3258,"Mead, Harris",Supporting
238938,282278,0,[],7625,Rob,Supporting
238939,282281,0,[],7625,Grimm,Supporting


In [27]:
dfCharacters.to_csv("../data/cleaned/anime_characters.csv", index=False)